In [26]:
import torch
import torch.nn as nn
import tiktoken
from torch.utils.data import Dataset, DataLoader

GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

torch.manual_seed(123)

In [27]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [28]:
def create_dataloader_v1(
    txt, batch_size=4, max_length=256,
    stride=128, shuffle=True, drop_last=True,
    num_workers=0):
    
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=True, #so that you dont have an incomplete chunk
        num_workers=num_workers,
    )

    return dataloader

In [47]:
class MultiHeadAttention(nn.Module):
    def __init__(
        self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False
    ):

        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads" #why

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # 768 // 12 = 64 per head
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_in, d_out) # mixes the heads back together
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)  # (b, num_tokens, d_out)
        values = self.W_value(x)
        queries = self.W_query(x)

        # split d_out into heads: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        # put heads next to batch: -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # one score matrix per head: (b, num_heads, num_tokens, num_tokens)
        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # (b, num_heads, num_tokens, head_dim) -> (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # glue the heads back together: -> (b, num_tokens, d_out)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)
        return context_vec

In [48]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim)) # learned, starts at 1
        self.shift = nn.Parameter(torch.zeros(emb_dim)) # learned, starts at 0

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False) # divide by n, like GPT2
        norm_x = (x-mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


In [49]:
ln = LayerNorm(emb_dim=5)
out_ln = ln(torch.randn(2, 5))

print(out_ln.mean(dim=-1, keepdim=True))
print(out_ln.var(dim=-1, keepdim=True, unbiased=False))

tensor([[2.3143e-08],
        [2.3842e-08]], grad_fn=<MeanBackward1>)
tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


In [50]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        # tanh approximation of GELU, the version GPT-2 used
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]), # 768 -> 3072
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]) #3072 -> 768
        )

    def forward(self, x):
        return self.layers(x)

In [51]:
ffn = FeedForward(GPT_CONFIG_124M)
print(ffn(torch.randn(2, 3, 768)).shape)

torch.Size([2, 3, 768])


In [52]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"]
        )

        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # sub-layer 1: attention, tokens exchange information
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        # sub-layer 2: feed-forward, each token processed on its own
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x
        

In [53]:
block = TransformerBlock(GPT_CONFIG_124M)
print(block(torch.rand(2, 4, 768)).shape)

torch.Size([2, 4, 768])


In [56]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"],
            cfg["vocab_size"],
            bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx) # (b, seq_len, 768)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device)) #(seq_len, 768)

        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x) # 12 blocks, shape unchanged
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits
        

In [73]:
def calc_loss_batch(
    input_batch, target_batch, model, device
):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss

def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0
    if len(data_loader) == 0:
        return float("nan")

    elif num_batches is None:
        num_batches = len(data_loader)

    else:
        num_batches = min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [74]:
def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):                 # one epoch = every batch once (9 here)
        model.train()                               # dropout on

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()                   # clear last batch's gradients
            loss = calc_loss_batch(input_batch, target_batch, model, device)   # 1 forward + 2 loss
            loss.backward()                         # 3 backward: fill every .grad
            optimizer.step()                        # 4 update: nudge every weight
            tokens_seen += input_batch.numel()      # 2 × 256 = 512 tokens per step
            global_step += 1

            if global_step % eval_freq == 0:        # every eval_freq steps, check progress
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        generate_and_print_sample(model, tokenizer, device, start_context)   # sample after each epoch

    return train_losses, val_losses, track_tokens_seen

In [75]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()                                    # dropout off for a fair measurement
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()                                   # back to training mode
    return train_loss, val_loss

In [76]:
def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]    # 256
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded, max_new_tokens=50, context_size=context_size)
    print(token_ids_to_text(token_ids, tokenizer).replace("\n", " "))
    model.train()

In [68]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

print("Characters:", len(text_data))
print("Tokens:", len(tokenizer.encode(text_data)))

# split on characters: first 90% train, last 10% validation
split_idx = int(0.9 * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

torch.manual_seed(123)
train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
)

print("\nTrain batches:")
for x, y in train_loader:
    print(x.shape, y.shape)
print("\nValidation batches:")
for x, y in val_loader:
    print(x.shape, y.shape)

Characters: 20479
Tokens: 5145

Train batches:
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])

Validation batches:
torch.Size([2, 256]) torch.Size([2, 256])


In [79]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps")
model.to(device)
model.eval()                                   # dropout off while measuring

torch.manual_seed(123)
with torch.no_grad():                          # measuring only, no gradients needed
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=2)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=2)
print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Training loss: 8.841863632202148
Validation loss: 9.12481689453125


In [83]:
import time
start_time = time.time()

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)          # fresh, untrained model
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

num_epochs = 10
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs,
    eval_freq=5,          # print train/val loss every 5 steps
    eval_iter=5,          # using up to 5 batches for each measurement
    start_context="Hi, my name is",
    tokenizer=tokenizer,
)

print(f"Training took {(time.time() - start_time) / 60:.2f} minutes.")

Ep 1 (Step 000000): Train loss 9.818, Val loss 9.923
Ep 1 (Step 000005): Train loss 8.067, Val loss 8.333
Hi, my name is.                                                 
Ep 2 (Step 000010): Train loss 6.621, Val loss 7.043
Ep 2 (Step 000015): Train loss 6.046, Val loss 6.597
Hi, my name is,,, and,,,,,,, and,.                                    
Ep 3 (Step 000020): Train loss 5.549, Val loss 6.483
Ep 3 (Step 000025): Train loss 5.449, Val loss 6.385
Hi, my name is to to the of the of the to the, and, and. G, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and,
Ep 4 (Step 000030): Train loss 5.007, Val loss 6.297
Ep 4 (Step 000035): Train loss 4.710, Val loss 6.277
Hi, my name is the picture.    "I had been the of the picture.           "I"I had the picture.      "I had been.    
Ep 5 (Step 000040): Train loss 4.146, Val loss 6.167
Hi, my name is the "Oh, and he had to the fact of the last I felt to me--and it the                            
Ep 6 (Step 00